In questo notebook provo a monitorare il comportamento effettivo di uno dei decoder addestrati, osservando come cambia lo stato dopo ogni step eseguito 

In [1]:
# import delle librerie
import numpy as np
import matplotlib.pyplot as plt
import torch

from Function_Library import *
from Environments import *
from Agent_Torch import *

import copy
import os
import time

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


Inizializzo agente ed env:

In [3]:
d = 5
p_phys = 0.005
p_meas = p_phys
error_model = "X"
volume_depth = 5
# rete neurale
cc_layers=[[64,3,2],[32,2,1],[32,2,1]]
ff_layers=[[512,0.2]]

# decoder statico
static_decoder_path = os.path.join(os.getcwd(),"referee_decoders/static_decoder_d5_X_p5.pth")
static_decoder = PyTorchStaticDecoder() 
static_decoder.load_state_dict(torch.load(static_decoder_path)) 
static_decoder.eval()
print('static_decoder caricato!')


static_decoder caricato!


In [15]:
# ambiente 
env = Surface_Code_Environment_Multi_Decoding_Cycles(
    d=d,
    p_phys=p_phys,
    p_meas=p_meas,
    error_model=error_model,
    volume_depth=volume_depth,
    static_decoder=static_decoder 
)

print(f'env inizializzato! \nerror model : {error_model}, p_err : {p_phys}')

env inizializzato! 
error model : X, p_err : 0.005


In [5]:
# policy allenata (agente che opera greedy)
input_shape = env.observation_space.shape
num_actions = env.num_actions
BEST_MODEL_PATH = r'runs\DDQN_d5_p0.005_20251014-081237\checkpoints\best_policy_net.pth'

policy_net = ConvNetRL(cc_layers, ff_layers, input_shape, num_actions, dueling=True).to(device)
policy_net.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
policy_net.eval()

policy_name = BEST_MODEL_PATH.split("\\")[1]
print(f'Policy inizializzata! \ncaricati i pesi della run {policy_name}')


Policy inizializzata! 
caricati i pesi della run DDQN_d5_p0.005_20251014-081237


Genero uno stato di partenza:

In [28]:
current_state, info = env.reset()

osservo lo stato nascosto (rappresenta l'errore effettivo che si è verificato, e su quale qubit fisico della griglia)
true_syndrome invece rappresenta la corrispondente sindrome generata, senza gli eventuali errori di misura che invece appariranno nello stato passato alla rete neurale 

In [30]:
print(env.hidden_state)
print(env.current_true_syndrome)


[[0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]]
[[0 0 0 1 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]]


board_state è lo stato che viene passato alla rete neurale: 5 strati di faulty syndrome e uno/due strati relativo allo storico di azioni compiute X (e Z eventualmente), il tutto codificato in una immagine (2d+1)x(2d+1)=11x11 che ne consente lo studio da parte della rete

la codifica utilizzata implementa alcuni "dummy" o "stabilizer orientation" index fissi che ne rendo meno intuitiva la lettura

In [33]:
# board_state passato all'env
print('1° strato faulty syndrome:')
print(env.board_state[0])
print('2° strato faulty syndrome:')
print(env.board_state[1])
print('3° strato faulty syndrome:')
print(env.board_state[2])
print('4° strato faulty syndrome:')
print(env.board_state[3])
print('5° strato faulty syndrome:')
print(env.board_state[4])
print('strato history action:')
print(env.board_state[5])

1° strato faulty syndrome:
[[0 1 0 1 0 1 1 1 0 1 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 0 0 0 1 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 1 0 1 0 1 0 1 0 1 0]]
2° strato faulty syndrome:
[[0 1 0 1 0 1 1 1 0 1 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 0 0 0 1 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 1 0 1 0 1 0 1 0 1 0]]
3° strato faulty syndrome:
[[0 1 0 1 0 1 1 1 0 1 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 0 0 0 0 1 0 0]
 [1 1 0 0 0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0 0 0 0 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 1 0 1 1 1 0 1 0 1 0]]
4° strato faulty syndrome:
[[0 1 0 1 0 1 1 1 0 1 0]
 [1 0 0 1 0 0 0 1 0 0 1]
 [0 0 0 0 1 0 

Sulla base dello stato attuale, seleziono l'azione greedy che dovrà essere compiuta sul qubit logico

In [34]:
state_tensor = torch.from_numpy(current_state).float().to(device).unsqueeze(0)
with torch.no_grad():
    q_values = policy_net(state_tensor).squeeze(0)

num_actions = q_values.size(-1)

action = q_values.argmax().item()

print(action)

2


Eseguo l'azione sullo stato e ne verifico l'esito sull'hidden_state

In [35]:
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

In [37]:
print(env.hidden_state)
print(env.current_true_syndrome)
print(f'reward ottenuta dall\'ultima azione: {reward}')

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]]
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]]
reward ottenuta dall'ultima azione: 0


L'agente è stato in grado di correggere uno degli errori che si erano verificati nella griglia del qubit logico.
Lo stato board_state che viene passato all'agente viene modificato solo nel history_layer :

In [38]:
print(env.board_state[5])

[[0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]]


A questo punto, ripeto il ciclo di selezione dell'azione tramite politica greedy ed esecuzione:

In [39]:
current_state = next_state

In [40]:
state_tensor = torch.from_numpy(current_state).float().to(device).unsqueeze(0)
with torch.no_grad():
    q_values = policy_net(state_tensor).squeeze(0)

num_actions = q_values.size(-1)

action = q_values.argmax().item()

print(action)

21


In [41]:
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

Osservo nuovamente lo stato nascosto:

In [42]:
print(env.hidden_state)
print(env.current_true_syndrome)
print(f'ricompensa ottenuta dall\'ultima azione: {reward}' )

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
ricompensa ottenuta dall'ultima azione: 0


In [43]:
print(env.board_state[5])

[[0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]]


Ripeto fino a che non ho corretto tutti gli errori su qubit fisici e ripristinato quindi lo stato logico:

In [44]:
current_state = next_state

In [45]:
state_tensor = torch.from_numpy(current_state).float().to(device).unsqueeze(0)
with torch.no_grad():
    q_values = policy_net(state_tensor).squeeze(0)

num_actions = q_values.size(-1)

action = q_values.argmax().item()

print(action)

13


In [46]:
next_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

In [47]:
print(env.hidden_state)
print(env.current_true_syndrome)
print(f'ricompensa ottenuta dall\'ultima azione: {reward}' )

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
ricompensa ottenuta dall'ultima azione: 1.0


In [48]:
print(env.board_state[5])

[[0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]]


Quando viene chiamata env.step(action), viene compiuta l'azione sullo stato nascosto dopodichè si verifica il nuovo stato:
- se lo stato è stato corretto, viene data ricompensa 1.0
- se lo stato non è stato corretto, ma è ancora decifrabile secondo il decoder referee, viene assegnata ricompensa zero
- se lo stato non è stato corretto e non è più decifrabile, termina l'episodio (con ricompensa zero)


In questo caso, l'agente è stato in grado di ripristinare lo stato del qubit logico con 3 azioni, e pertanto ottiene come ricompensa 1 solo per l'ultima azione.